<a href="https://colab.research.google.com/github/emmanuelmassawe/breast-cancer-predictions-with-pytorch/blob/main/breast_cancer_predictions_with_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# importing the necessary libraries

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset,DataLoader

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


loaing the dataset

In [16]:
data = load_breast_cancer()
X, y = data.data, data.target

we train_test and split the dataset

In [17]:
X_train,X_test ,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42 , stratify=y)


we check the shape of the data

In [18]:
print(X_train.shape)
print(X_test.shape)

(455, 30)
(114, 30)


we perform feature scaling


In [19]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

we change to tensor arrays

In [20]:
X_train = torch.tensor(X_train,dtype=torch.float32).to(device)
X_test = torch.tensor(X_test,dtype=torch.float32).to(device)

y_train = torch.tensor(y_train, dtype = torch.long).to(device)
y_test = torch.tensor(y_test, dtype = torch.long).to(device)



we create a dataset and a data loader

In [21]:
train_dataset = TensorDataset(X_train,y_train)
test_dataset = TensorDataset(X_test,y_test)

train_loader = DataLoader(train_dataset, batch_size= 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size= 32, shuffle = False)

creating the Neural Network Architecture


In [26]:
class Nn(nn.Module):
  def __init__(self):
    super().__init__()

    self.network = nn.Sequential(
        nn.Linear(30,64),
        nn.ReLU(),
        nn.Linear(64,32),
        nn.ReLU(),
        nn.Linear(32,2)
    )
  def forward(self,x):
    return self.network(x)

model = Nn().to(device)
print(model)

Nn(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=2, bias=True)
  )
)


Apply the loss function

In [31]:
loss_fn = nn.CrossEntropyLoss().to(device)

we apply an optimizer

In [32]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 0.001
)

train the model

In [34]:
epoch = 20

for i in range(epoch):
  model.train()
  for X_batch,y_batch in train_loader:
    optimizer.zero_grad()
    y_pred = model(X_batch)
    loss = loss_fn(y_pred,y_batch)


    loss.backward()
    optimizer.step()

    if (i+1) % 10 == 0:
      print(f"Epoch {i+1}/{epoch}, Loss: {loss.item():.4f}")



Epoch 10/20, Loss: 0.0000
Epoch 10/20, Loss: 0.0010
Epoch 10/20, Loss: 0.0002
Epoch 10/20, Loss: 0.0005
Epoch 10/20, Loss: 0.0001
Epoch 10/20, Loss: 0.0003
Epoch 10/20, Loss: 0.0001
Epoch 10/20, Loss: 0.0000
Epoch 10/20, Loss: 0.0001
Epoch 10/20, Loss: 0.0000
Epoch 10/20, Loss: 0.0187
Epoch 10/20, Loss: 0.0001
Epoch 10/20, Loss: 0.0001
Epoch 10/20, Loss: 0.0004
Epoch 10/20, Loss: 0.0012
Epoch 20/20, Loss: 0.0001
Epoch 20/20, Loss: 0.0001
Epoch 20/20, Loss: 0.0003
Epoch 20/20, Loss: 0.0000
Epoch 20/20, Loss: 0.0003
Epoch 20/20, Loss: 0.0001
Epoch 20/20, Loss: 0.0000
Epoch 20/20, Loss: 0.0008
Epoch 20/20, Loss: 0.0000
Epoch 20/20, Loss: 0.0000
Epoch 20/20, Loss: 0.0001
Epoch 20/20, Loss: 0.0187
Epoch 20/20, Loss: 0.0001
Epoch 20/20, Loss: 0.0001
Epoch 20/20, Loss: 0.0000


model evaluation


In [37]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
  for X_batch,y_batch in test_loader:
    y_pred = model(X_batch)
    pred = torch.argmax(y_pred , dim = 1)

    correct += (pred == y_batch).sum().item()
    total += y_batch.size(0)

accuracy = correct / total
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 95.61%


predicting the new breast cancer
